In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from datasets import Dataset
import regex
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, TrainingArguments, Trainer, DataCollatorForSeq2Seq
import evaluate

c:\Users\User\.conda\envs\torch_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Load and Inspect dataset

In [2]:
df = pd.read_excel("dataset/english_to_urdu_dataset.xlsx")

In [3]:
df.head()

,eng,urdu
0,the book of the generation of jesus christ th...,یسوع مسیح ابن داود ابن ابرہام کا نسب نامہ
1,abraham begat isaac and isaac begat jacob an...,ابراہام سے اضحاق پیدا ہوا اور اضحاق سے یعقوب پ...
2,and judas begat phares and zara of thamar and...,اور یہوداہ سے فارص اور زارح تمر سے پیدا ہوئے ا...
3,and aram begat aminadab and aminadab begat na...,اور رام سے عمینداب پیدا ہوا اور عمینداب سے نحس...
4,and salmon begat booz of rachab and booz bega...,اور سلمون سے بوعز راحب سے پیدا ہوا اور بوعز سے...


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9103 entries, 0 to 9102
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   eng     9103 non-null   str  
 1   urdu    9102 non-null   str  
dtypes: str(2)
memory usage: 2.5 MB


Inspection shows:
1. There is one null value
2. English text have some extra spaces
3. No URLs, HTML tags or escape sequences detected.

# Pre processing

### Remove null values

In [5]:
df.isnull().sum()

eng     0
urdu    1
dtype: int64

In [6]:
df = df.dropna()

### Remove duplicates

In [7]:
df[df.duplicated()]

,eng,urdu
361,that it might be fulfilled which was spoken by...,تاکہ جو یسعیاہ نبی کی معرفت کہا گیا تھا وہ پور...
1439,where their worm dieth not and the fire is no...,[ جہاں ان کا کیڑا نہیں مرتا اور آگ نہیں بجھتی ] ۔
2073,and blessed is he whosoever shall not be offe...,اور مبارک ہے وہ جو میرے سبب سے ٹھوکر نہ کھائے ۔
2270,for every one that asketh receiveth and he th...,کیونکہ جو کوئی مانگتا ہے اسے ملتا ہے اور جو ڈھ...
2378,i tell you nay but except ye repent ye sha...,میں تم سے کہتا ہوں کہ نہیں بلکہ اگر تم توبہ نہ...
6218,grace be unto you and peace from god our fat...,ہمارے باپ خدا اور خداوند یسوع مسیح کی طرف سے ت...
7607,he that hath an ear let him hear what the spi...,جس کے کان ہوں وہ سنے کہ روح کلیسیاؤں سے کیا ف...
7614,he that hath an ear let him hear what the spi...,جس کے کان ہوں وہ سنے کہ روح کلیسیاؤں سے کیا ف...
7623,he that hath an ear let him hear what the spi...,جس کے کان ہوں وہ سنے کہ روح کلیسیاؤں سے کیا ف...


The English text and translation text are exact duplicates. So it's safe to remove them.

In [8]:
df = df.drop_duplicates()

### Splitting

In [9]:
df = df.sample(5000, random_state=42)
train_dataset, test_dataset = train_test_split(df, test_size=0.15, random_state=42)
val_dataset, test_dataset = train_test_split(test_dataset, test_size=0.2, random_state=42)

### Cleaning

In [10]:
def clean_text(text):
    text = regex.sub("\s+", " ", text)
    return text.strip()

In [11]:
train_dataset["eng"] = train_dataset["eng"].map(clean_text)
train_dataset["urdu"] = train_dataset["urdu"].map(clean_text)
val_dataset["eng"] = val_dataset["eng"].map(clean_text)
val_dataset["urdu"] = val_dataset["urdu"].map(clean_text)
test_dataset["eng"] = test_dataset["eng"].map(clean_text)
test_dataset["urdu"] = test_dataset["urdu"].map(clean_text)

### Tokenization

In [12]:
train_dataset = Dataset.from_pandas(train_dataset)
val_dataset = Dataset.from_pandas(val_dataset)

In [13]:
tokenizer = AutoTokenizer.from_pretrained("google/mt5-small")

In [16]:
def tokenize(data):
    data["eng"] = ["translate english to urdu " + element for element in data["eng"]]
    inputs = tokenizer(data["eng"], max_length=256, truncation=True)
    outputs = tokenizer(data["urdu"], max_length=256, truncation=True)
    inputs["labels"] = outputs["input_ids"]
    return inputs

In [17]:
train_dataset = train_dataset.map(
    tokenize,
    batched=True
)
val_dataset = val_dataset.map(
    tokenize,
    batched=True
)

Map: 100%|██████████| 600/600 [00:00<00:00, 14808.91 examples/s]


# Model training & evaluation

### Loading the model

In [18]:
model  = AutoModelForSeq2SeqLM.from_pretrained("google/mt5-small") 

Loading weights: 100%|██████████| 192/192 [00:00<00:00, 16004.34it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [21]:
# adding padding in tokens
data_collator = DataCollatorForSeq2Seq(
    model=model,
    tokenizer=tokenizer
)

In [ ]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
model.to(device)

MT5ForConditionalGeneration(
  (shared): Embedding(250112, 512)
  (encoder): MT5Stack(
    (embed_tokens): Embedding(250112, 512)
    (block): ModuleList(
      (0): MT5Block(
        (layer): ModuleList(
          (0): MT5LayerSelfAttention(
            (SelfAttention): MT5Attention(
              (q): Linear(in_features=512, out_features=384, bias=False)
              (k): Linear(in_features=512, out_features=384, bias=False)
              (v): Linear(in_features=512, out_features=384, bias=False)
              (o): Linear(in_features=384, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 6)
            )
            (layer_norm): MT5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): MT5LayerFF(
            (DenseReluDense): MT5DenseGatedActDense(
              (wi_0): Linear(in_features=512, out_features=1024, bias=False)
              (wi_1): Linear(in_features=512, out_features=1024, bias=False)
          

### Fine tuning

In [ ]:
# for google collab
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    # output_dir="/content/drive/MyDrive/results" # for google collab
    num_train_epochs=10,
    warmup_steps=500,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    per_device_eval_batch_size=2, # small batch size cause out of memory error was occuring
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4, # to handle memory during gradient calculation
    fp16=True, # for efficient memory usage by reducing size of number from 32 to 16 bits
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator
)

In [ ]:
trainer.train()

### Saving and loading the best model

In [ ]:
tokenizer.save("./saved_translation_model")
model.save("./saved_translation_model")
# model.save_pretrained("/content/drive/MyDrive/saved_summary_model") # for google collab
# tokenizer.save_pretrained("/content/drive/MyDrive/saved_summary_model")

In [24]:
model = AutoModelForSeq2SeqLM.from_pretrained("./saved_translation_model")
tokenizer = AutoTokenizer.from_pretrained("./saved_translation_model")

Loading weights: 100%|██████████| 190/190 [00:00<00:00, 1097.48it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


### Testing

In [ ]:
def translate(text):
    input_tokens = tokenizer(
        text,
        max_length=256,
        truncation=True,
        return_tensors="pt",
        early_stopping=True
    )
    output_tokens = model.generate(
        input_tokens["input_ids"],
        attention_mask=input_tokens["attention_mask"],
        max_length=256,
    )
    return tokenizer.decode(
        output_tokens[0],
        skip_special_tokens=True
    )

In [54]:
text="and he opened his mouth  and taught them  saying"

In [55]:
translate(text)

'اور وہ اس کے ہاتھ سے کہا کہ وہ کہا کہ وہ کہا کہ وہ کہا کہ وہ کہا کہ وہ کہا کہ وہ کہا کہ وہ کہا کہ وہ کہا کہ وہ کہا کہ وہ کہا کہ وہ کہا کہ وہ کہا کہ وہ کہا کہ وہ کہا کہ وہ کہا کہ وہ کہا کہ وہ کہا کہ وہ کہا کہ وہ کہا کہ وہ کہا کہ وہ کہا کہ وہ کہا کہ وہ کہا کہ وہ کہا کہ وہ کہا کہ وہ کہا کہ وہ کہا کہ وہ کہا کہ وہ کہا کہ وہ کہا کہ وہ کہا کہ وہ کہا کہ وہ کہا کہ وہ کہا کہ وہ کہا کہ وہ کہا کہ وہ کہا کہ وہ کہا کہ وہ کہا کہ وہ کہا کہ وہ کہا کہ وہ کہا کہ وہ کہا کہ وہ کہا کہ وہ کہا کہ وہ کہا کہ وہ کہا کہ وہ کہا کہ وہ '

### Evaluation

In [57]:
actual_outputs = []
predicted_outputs = []
counter = 1

for data in test_dataset.itertuples():
    print("Iteration: ", counter)
    counter += 1

    input = data.eng
    actual_output = data.urdu
    predicted_output = translate(input)

    actual_outputs.append(actual_output)
    predicted_outputs.append(predicted_output)

Iteration:  1
Iteration:  2
Iteration:  3
Iteration:  4
Iteration:  5
Iteration:  6
Iteration:  7
Iteration:  8
Iteration:  9
Iteration:  10
Iteration:  11
Iteration:  12
Iteration:  13
Iteration:  14
Iteration:  15
Iteration:  16
Iteration:  17
Iteration:  18
Iteration:  19
Iteration:  20
Iteration:  21
Iteration:  22
Iteration:  23
Iteration:  24
Iteration:  25
Iteration:  26
Iteration:  27
Iteration:  28
Iteration:  29
Iteration:  30
Iteration:  31
Iteration:  32
Iteration:  33
Iteration:  34
Iteration:  35
Iteration:  36
Iteration:  37
Iteration:  38
Iteration:  39
Iteration:  40
Iteration:  41
Iteration:  42
Iteration:  43
Iteration:  44
Iteration:  45
Iteration:  46
Iteration:  47
Iteration:  48
Iteration:  49
Iteration:  50
Iteration:  51
Iteration:  52
Iteration:  53
Iteration:  54
Iteration:  55
Iteration:  56
Iteration:  57
Iteration:  58
Iteration:  59
Iteration:  60
Iteration:  61
Iteration:  62
Iteration:  63
Iteration:  64
Iteration:  65
Iteration:  66
Iteration:  67
Iter

In [58]:
bleu = evaluate.load("bleu")
bleu_score = bleu.compute(
    predictions=predicted_outputs,
    references=actual_outputs
)
print(f"BELU score: {bleu_score}")

BELU score: {'bleu': 0.004176290931934678, 'precisions': [0.050312754963285286, 0.007627975369910854, 0.0013978194017332962, 0.0005670541536716756], 'brevity_penalty': 1.0, 'length_ratio': 3.2655417406749554, 'translation_length': 11031, 'reference_length': 3378}
